# Review Assignment NLP for Quiz



In [94]:
import pandas as pd # untuk read csv
import string # untuk menghapus tanda baca
import pickle # untuk menyimpan model yang sudah dilatih
import os # untuk read file
import nltk # untuk natural language processing
from nltk.tokenize import word_tokenize # untuk tokenisasi (kalimat -> kata)
from nltk.corpus import stopwords # untuk menghapus stopwords
from nltk.stem import SnowballStemmer # untuk stemming (mengubah kata ke bentuk dasar)
from nltk.stem import WordNetLemmatizer # untuk lemmatization (mengubah kata ke bentuk dasar)
from nltk.corpus import wordnet # untuk cari sinonim dan antonim
from nltk.tag import pos_tag # untuk part of speech tagging
from nltk.probability import FreqDist # untuk menghitung frekuensi kata
from nltk.chunk import ne_chunk # untuk named entity recognition
from nltk.classify import NaiveBayesClassifier, accuracy # untuk klasifikasi menggunakan Naive Bayes dan akurasinya

MODEL_PATH = 'ReviewCase/model.pickle'
CSV_PATH = 'ReviewCase/financial_dataset.csv'

# LAB PT 1 (Tokenizing, Stopwords, Removed Punctuation)

In [95]:
# ambil satu kalimat pada dataset

df = pd.read_csv(CSV_PATH)
print(df.head())
sentence = df['Statement'][0]

print("\n - Origin Sentence")
print("Sentence:", sentence)

# Lowercase
print("\n - Lowercase")
sentence = sentence.lower()
print("Lowercase Sentence:", sentence)

# word_tokenize
print("\n - Tokenization")
word_list = word_tokenize(sentence) # sent_tokenize untuk kalimat, word_tokenize untuk kata
print("Word List:", word_list)

# Stopwords
print("\n - Stopwords List")
eng_stopwords = stopwords.words('english')
print("Stopwords:", eng_stopwords)
# Remove Stopwords
removed_stopwords = [word for word in word_list if word not in eng_stopwords]
print("\n - Removed Stopwords")
print("Removed Stopwords:", removed_stopwords)

# Punctuation
print("\n - Punctuation List")
punctuation = string.punctuation
print("Punctuation:", punctuation)

# Remove Punctuation
removed_punctuation = [word for word in removed_stopwords if word not in punctuation]
print("\n - Removed Punctuation")
print("Removed Punctuation:", removed_punctuation)

# Remove Number
print("\n - Removed Number")
removed_number = [word for word in removed_punctuation if word.isalpha()]
print("Removed Number:", removed_number)

                                           Statement Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive
3    $SPY wouldn't be surprised to see a green close  positive
4  Shell's $70 Billion BG Deal Meets Shareholder ...  negative

 - Origin Sentence
Sentence: The GeoSolutions technology will leverage Benefon 's GPS solutions by providing Location Based Search Technology , a Communities Platform , location relevant multimedia content and a new and powerful commercial model .

 - Lowercase
Lowercase Sentence: the geosolutions technology will leverage benefon 's gps solutions by providing location based search technology , a communities platform , location relevant multimedia content and a new and powerful commercial model .

 - Tokenization
Word List: ['the', 'geosolutions', 'technology', 'will', 'leverage', 'benefon', "'s", 'gps', 'solutions

# LAB PT 2 (Stemming, Lemmatization, Wordnet)

In [96]:
stemmer = SnowballStemmer('english') # Stemming klo indo jadinya 'Indonesian'

word_list = [word for word in removed_number]

# Original Word List
print("\n - Original Word List")
print("Original Word List:", word_list)

# Stemming
print("\n - Stemming")
stemming = [stemmer.stem(word) for word in word_list]
print("Stemming Result:", stemming)

# Lemmatization
print("\n - Lemmatization")
lemmatizer = WordNetLemmatizer() # Lemmatization menggunakan WordNetLemmatizer
lemmatizing = [lemmatizer.lemmatize(word) for word in word_list]
print("Lemmatization Result:", lemmatizing)

# Coba kebentuk kata lain
print("\n - Lemmatization with POS Tagging")
adjective = [lemmatizer.lemmatize(word, 'a') for word in word_list] # a untuk adjective
adverb = [lemmatizer.lemmatize(word, 'r') for word in word_list] # r untuk adverb
verb = [lemmatizer.lemmatize(word, 'v') for word in word_list] # v untuk verb
noun = [lemmatizer.lemmatize(word, 'n') for word in word_list] # n untuk noun
print("Adjective:", adjective)
print("Adverb:", adverb)
print("Verb:", verb)
print("Noun:", noun)

# Mencari sinonim dan antonim dari sebuah kata
print("\n - Synonym and Antonym")
for word in lemmatizing:
    synonyms = []
    antonyms = []

    synsets = wordnet.synsets(word)
    for syn in synsets:
        for lemma in syn.lemmas():
            synonyms.append(lemma.name())
            for ant in lemma.antonyms():
                antonyms.append(ant.name())
    
    print("Word:", word)
    if synonyms:
        print("Synonyms:", str(set(synonyms[:5]))) # Ambil 5 sinonim pertama
    else:
        print("Synonyms: None")
    if antonyms:
        print("Antonyms:", str(set(antonyms[:5]))) # Ambil 5 antonim pertama
    else:
        print("Antonyms: None")
    
    print()




 - Original Word List
Original Word List: ['geosolutions', 'technology', 'leverage', 'benefon', 'gps', 'solutions', 'providing', 'location', 'based', 'search', 'technology', 'communities', 'platform', 'location', 'relevant', 'multimedia', 'content', 'new', 'powerful', 'commercial', 'model']

 - Stemming
Stemming Result: ['geosolut', 'technolog', 'leverag', 'benefon', 'gps', 'solut', 'provid', 'locat', 'base', 'search', 'technolog', 'communiti', 'platform', 'locat', 'relev', 'multimedia', 'content', 'new', 'power', 'commerci', 'model']

 - Lemmatization
Lemmatization Result: ['geosolutions', 'technology', 'leverage', 'benefon', 'gps', 'solution', 'providing', 'location', 'based', 'search', 'technology', 'community', 'platform', 'location', 'relevant', 'multimedia', 'content', 'new', 'powerful', 'commercial', 'model']

 - Lemmatization with POS Tagging
Adjective: ['geosolutions', 'technology', 'leverage', 'benefon', 'gps', 'solutions', 'providing', 'location', 'based', 'search', 'techno

# LAB PT 3 (POS Tagging and Frequency Distribution)


In [97]:
word_list = [word for word in removed_number]

# Original Word List
print("\n - Original Word List")
print("Original Word List:", word_list)

# POST Tagging
print("\n - POS Tagging")
tagged = pos_tag(word_list)
print("POS Tagging Result:", tagged)

# Named Entity Recognition
print("\n - Named Entity Recognition")
ner = ne_chunk(tagged)
print("NER: ", ner)

# Frequency Distribution
print("\n - Frequency Distribution")
freq_dist = FreqDist(word_list)
for word, count in freq_dist.most_common(5):
    print("word:", word, ", count:", count)


 - Original Word List
Original Word List: ['geosolutions', 'technology', 'leverage', 'benefon', 'gps', 'solutions', 'providing', 'location', 'based', 'search', 'technology', 'communities', 'platform', 'location', 'relevant', 'multimedia', 'content', 'new', 'powerful', 'commercial', 'model']

 - POS Tagging
POS Tagging Result: [('geosolutions', 'NNS'), ('technology', 'NN'), ('leverage', 'NN'), ('benefon', 'VBD'), ('gps', 'JJ'), ('solutions', 'NNS'), ('providing', 'VBG'), ('location', 'NN'), ('based', 'VBN'), ('search', 'NN'), ('technology', 'NN'), ('communities', 'NNS'), ('platform', 'VBP'), ('location', 'NN'), ('relevant', 'JJ'), ('multimedia', 'NN'), ('content', 'JJ'), ('new', 'JJ'), ('powerful', 'JJ'), ('commercial', 'JJ'), ('model', 'NN')]

 - Named Entity Recognition
NER:  (S
  geosolutions/NNS
  technology/NN
  leverage/NN
  benefon/VBD
  gps/JJ
  solutions/NNS
  providing/VBG
  location/NN
  based/VBN
  search/NN
  technology/NN
  communities/NNS
  platform/VBP
  location/NN
  r

# LAB PT 4 (Sentiment Classification with Naive Bayes)


In [98]:
# 1. Preprocessing data -> membersihkan data
# 2. Feature extraction -> mengekstrak fitur dari data menjadi data biner (False/True)
# 3. Data Splitting -> 80% training, 20% testing
# 4. Melatih model Naive Bayes

In [99]:
df = pd.read_csv(CSV_PATH)
print(df.head())
print(df['Sentiment'].value_counts())

                                           Statement Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive
3    $SPY wouldn't be surprised to see a green close  positive
4  Shell's $70 Billion BG Deal Meets Shareholder ...  negative
Sentiment
positive    1852
negative     860
Name: count, dtype: int64


In [100]:
# 1. Preprocessing Data

def get_tag(tag):
    if tag.startswith('j'):
        return 'a'
    elif tag.startswith('r') or tag.startswith('v') or tag.startswith('n'):
        return tag[0]
    else:
        return 'n'

def lemmatizing(stemmed):
    lemmatizer = WordNetLemmatizer()
    lemmatized = []
    tagged = pos_tag(stemmed)

    for word, tag in tagged:
        label = get_tag(tag.lower())
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemmatized.append(result)
        else:
            result = lemmatizer.lemmatize(word)
            lemmatized.append(result)
    
    return lemmatized
    

def preprocess_sentence(sentence):
    punctuation = string.punctuation
    eng_stopwords = stopwords.words('english')
    stemmer = SnowballStemmer('english')

    word_list = word_tokenize(sentence)
    word_list = [word.lower() for word in word_list]
    removed_stopwords = [word for word in word_list if word not in eng_stopwords]
    removed_punc = [word for word in removed_stopwords if word not in punctuation]
    stemmed = [stemmer.stem(word) for word in removed_punc]

    lemmatized = lemmatizing(stemmed)

    return lemmatized


In [101]:
# 2. Feature extraction

x = df['Statement']
y = df['Sentiment']

all_comment = ' '.join(x)
all_token = preprocess_sentence(all_comment)

freq_dist = FreqDist(all_token)

print(freq_dist.most_common(10))

[('eur', 747), ("'s", 474), ('mn', 459), ('profit', 387), ('sale', 338), ('compani', 321), ('oper', 305), ('net', 297), ('say', 296), ('finnish', 267)]


In [102]:
# fungsi ekstraksi features
def extract_features(statement):
    features = {}
    for word in freq_dist.keys():
        features[word] = (word in statement)
    return features

# Membuat feature sets, bakalan ngehasilin berupa list of tuple
feature_sets = []
for(statement, sentiment) in zip(x, y):
    features = extract_features(preprocess_sentence(statement))
    feature_sets.append((features, sentiment))

print(feature_sets[0])


({'geosolut': True, 'technolog': True, 'leverag': True, 'benefon': True, "'s": True, 'gps': True, 'solut': True, 'provid': True, 'locat': True, 'base': True, 'search': True, 'communiti': True, 'platform': True, 'relev': True, 'multimedia': True, 'content': True, 'new': True, 'power': True, 'commerci': True, 'model': True, 'esi': False, 'low': False, '1.50': False, '2.50': False, 'bk': False, 'real': False, 'possibl': False, 'last': False, 'quarter': False, '2010': False, 'componenta': False, 'net': False, 'sale': False, 'doubl': False, 'eur131m': False, 'eur76m': False, 'period': False, 'year': False, 'earlier': False, 'move': False, 'zero': False, 'pre-tax': False, 'profit': False, 'loss': False, 'eur7m': False, 'spi': False, 'would': False, "n't": False, 'surpris': False, 'see': False, 'green': False, 'close': False, 'shell': False, '70': False, 'billion': False, 'bg': False, 'deal': False, 'meet': False, 'sharehold': False, 'skeptic': False, 'ssh': False, 'communic': False, 'secur':

In [103]:
# 3. Data splitting
train_count = int(len(feature_sets) * 0.8) # jika ingin diubah splittingnya
train_set = feature_sets[:train_count]
test_set = feature_sets[train_count:]

In [ ]:
# 4. Melatih model Naive Bayes

classifier = NaiveBayesClassifier.train(train_set)

test_accuracy = accuracy(classifier, test_set)
print("Test Accuracy: ", test_accuracy*100, "%")

# Write
with open(MODEL_PATH, "wb") as f:
    pickle.dump(classifier, f)

# Read
with open(MODEL_PATH, "rb") as f:
    classifier = pickle.load(f)

KeyboardInterrupt: 

In [ ]:
text = input("Input text: ")

preprocessed_text = preprocess_sentence(text)
features = extract_features(preprocessed_text)
predicted_emotion = classifier.classify(features)
print(predicted_emotion)



negative


# Additional

In [ ]:
from IPython.display import clear_output

# Looping menu sederhana (infinite loop sampai pilih exit)
while True:
    clear_output(True)  # supaya output tidak menumpuk ke bawah
    print("=== MENU SENTIMENT ANALYSIS ===")
    print("1. Prediksi sentiment")
    print("2. Lihat akurasi model")
    print("3. Exit")

    pilihan = input("Pilih menu (0/1/2): ").strip()

    if pilihan == "1":
        text = input("Input text: ")
        preprocessed_text = preprocess_sentence(text)
        features = extract_features(preprocessed_text)
        predicted_emotion = classifier.classify(features)
        print("Hasil prediksi:", predicted_emotion)
        input("\nTekan Enter untuk kembali ke menu...")

    elif pilihan == "2":
        if 'test_accuracy' in globals():
            print(f"Akurasi model: {test_accuracy * 100:.2f}%")
        else:
            print("Akurasi belum tersedia. Jalankan cell training terlebih dahulu.")
        input("\nTekan Enter untuk kembali ke menu...")

    elif pilihan == "3":
        clear_output(True)
        print("Program selesai. Loop dihentikan.")
        break

    else:
        print("Pilihan tidak valid. Coba lagi.")
        input("\nTekan Enter untuk kembali ke menu...")

=== MENU SENTIMENT ANALYSIS ===
1. Prediksi sentiment
2. Lihat akurasi model
3. Exit
Program selesai. Loop dihentikan.
